# Notebook 07 — Specialising the Selected Model

**AI Interview Assistant · Machine Learning Pipeline, Stage 7 of 9**

---

## Purpose

Take the architecture Stage 6 selected and train it further on the **specific
task** the runtime needs: generating an interview question when given a domain
and a difficulty.

## Why a second stage of training

Stage 5 trained a general language model over question text. The runtime does
not want free-form text — it wants a question that matches a *requested* domain
and difficulty. Specialisation changes the training objective to teach exactly
that conditioning.

**The format the model learns:**

```
<DOMAIN: SQL> <DIFFICULTY: Intermediate> What is the difference between an
inner join and a left join?
```

The model sees the control tokens first, so at inference time supplying them
steers what it generates.

## This is continued training, not adapter fine-tuning

The project uses **no LoRA, no adapters, no external fine-tuning frameworks**.
Every weight in the model was created by this project and is updated directly.
Step 1 asserts the checkpoint being loaded is our own Stage 5 output.

## Guarding against catastrophic forgetting

Continued training at the original learning rate can destroy what the base model
learned. Three countermeasures:

| Countermeasure | Purpose |
|---|---|
| Learning rate reduced to **⅕** of Stage 5's | small, careful updates |
| Validation checked every epoch, with early stopping | halt before degradation |
| **Base vs specialised comparison on validation** | prove the change helped |

If specialisation makes validation loss worse, Step 6 says so and the base model
is what Stage 8 evaluates. A stage that cannot fail is not a test.

## Outputs

- `checkpoints/specialized_model/`
- `reports/specialization_report.json`, `reports/figures/07_*.png`

---

In [ ]:
NOTEBOOK_ID = 7

# ─────────────────────────────────────────────────────────────────────────────
# Step 0 — Environment bootstrap
#
# Locates the project workspace so this notebook runs unchanged in Google Colab,
# a local Jupyter server, or VS Code. Every later step resolves its paths from
# WORKSPACE_DIR, so nothing below depends on where the notebook was opened.
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def locate_workspace() -> Path:
    """Return the ml-service directory, whatever environment we are in."""
    # 1. Google Colab: mount Drive so checkpoints survive a runtime restart.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        ws = Path("/content/drive/MyDrive/ai-interview-system/ml-service")
        ws.mkdir(parents=True, exist_ok=True)
        print("Environment      : Google Colab (Drive mounted)")
        return ws
    except ImportError:
        pass

    # 2. Local: walk up from the notebook until we find the ml-service root,
    #    identified by the dataset directory it must contain.
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            print("Environment      : local")
            return candidate
    print("Environment      : local (fallback to cwd)")
    return here

WORKSPACE_DIR = locate_workspace()
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# Canonical paths used across all nine notebooks.
RAW_DIR       = WORKSPACE_DIR / "dataset" / "raw"
PROCESSED_DIR = WORKSPACE_DIR / "dataset" / "processed"
QG_DIR        = PROCESSED_DIR / "question_generator"
SPLIT_DIR     = PROCESSED_DIR / "splits"
TOKENIZER_DIR = WORKSPACE_DIR / "tokenizer"
CKPT_DIR      = WORKSPACE_DIR / "checkpoints"
MODEL_DIR     = WORKSPACE_DIR / "models"
REPORTS_DIR   = WORKSPACE_DIR / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"

for d in (RAW_DIR, PROCESSED_DIR, SPLIT_DIR, TOKENIZER_DIR, CKPT_DIR,
          MODEL_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace        : {WORKSPACE_DIR}")
print(f"Python           : {sys.version.split()[0]}")
print(f"Run started      : {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

---

## Step 0b — Figure and statistics conventions

One style definition serves every figure in the nine-notebook pipeline, so
charts are directly comparable when placed side by side in the write-up.

Three conventions are fixed here:

1. **A colour-blind-safe categorical palette** — the same six colours, in the
   same order, wherever a chart encodes categories.
2. **Automatic figure export** — `save_figure()` writes every figure to
   `reports/figures/` at 200 dpi with a numbered filename, and prints its
   caption, so figures can be cited as *Figure N.k* in the dissertation.
3. **A single summary-statistics function** — `describe_series()` reports
   n, mean, sd, the five-number summary, skewness and kurtosis in a fixed
   order for every variable, so distributions are described consistently.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 0b — Plotting conventions
#
# One style definition for every figure in the pipeline, so figures across the
# nine notebooks are directly comparable in the dissertation. Every figure is
# also saved to reports/figures/ at 200 dpi, ready to drop into the write-up.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#444444",
    "grid.alpha": 0.3,
    "legend.frameon": True,
    "figure.autolayout": False,
})

# Colour-blind-safe categorical palette, used consistently for every chart.
PALETTE = ["#3B6FD4", "#E1893B", "#3EA37A", "#C4576B", "#7B5EA7", "#8C7B68"]
sns.set_palette(PALETTE)

_figure_index = {"n": 0}

def save_figure(fig, slug: str, caption: str = "") -> Path:
    """Save a figure with a numbered filename and print its caption."""
    _figure_index["n"] += 1
    n = _figure_index["n"]
    path = FIGURES_DIR / f"{NOTEBOOK_ID:02d}_fig{n:02d}_{slug}.png"
    fig.savefig(path)
    label = f"Figure {NOTEBOOK_ID}.{n}"
    if caption:
        print(f"{label}: {caption}")
    print(f"           saved -> {path.relative_to(WORKSPACE_DIR)}")
    return path

def describe_series(series: pd.Series, name: str) -> pd.Series:
    """Summary statistics reported in a consistent order for every variable."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "min": s.min(),
        "q1": s.quantile(0.25),
        "median": s.median(),
        "q3": s.quantile(0.75),
        "max": s.max(),
        "skew": s.skew(),
        "kurtosis": s.kurtosis(),
    }, name=name)

print("Plot style       : configured")
print(f"Figure output    : {FIGURES_DIR.relative_to(WORKSPACE_DIR)}")
print(f"Palette          : {len(PALETTE)} colour-blind-safe categories")

---

## Step 1 — Load the winning checkpoint

The provenance assertion is the point of this step: the loaded weights must come
from a Stage 5 checkpoint directory inside this project.

In [ ]:
import math
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformer_scratch import (
    CustomBPETokenizer, build_candidate_model,
    save_checkpoint, load_checkpoint,
)

SELECTION_FILE = REPORTS_DIR / "model_selection.json"
assert SELECTION_FILE.exists(), (
    f"{SELECTION_FILE.name} missing — run Notebook 06 first."
)
selection = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))
selected = selection["selected"]

CANDIDATE_ID = selected["candidate_id"]
BASE_CKPT_DIR = WORKSPACE_DIR / selected["checkpoint"]

print("SELECTED ARCHITECTURE (from Stage 6)")
print("=" * 74)
print(f"  Candidate        : {selected['label']}")
print(f"  Identifier       : {CANDIDATE_ID}")
print(f"  Composite score  : {selected['composite_score']:.2f}/100")
print(f"  Base val loss    : {selected['val_loss']:.4f}")
print(f"  Parameters       : {selected['parameters'] / 1e6:.2f}M")
print(f"  Checkpoint       : {BASE_CKPT_DIR.relative_to(WORKSPACE_DIR)}")
print("=" * 74)

# ── Provenance assertion: these must be OUR weights ────────────────────────
assert BASE_CKPT_DIR.exists(), (
    f"Checkpoint directory {BASE_CKPT_DIR} not found. Re-run Notebook 05."
)
assert (BASE_CKPT_DIR / "checkpoint.pt").exists(), (
    "checkpoint.pt not found — nothing to specialise."
)
assert CKPT_DIR in BASE_CKPT_DIR.parents, (
    f"The checkpoint must live under this project's checkpoints/ directory. "
    f"Found {BASE_CKPT_DIR}, which is outside {CKPT_DIR}. Specialising "
    f"external weights would violate the zero-pretrained-model policy."
)
print("\nProvenance assertion passed: the weights are this project's own "
      "Stage 5 output.")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = CustomBPETokenizer.load(TOKENIZER_DIR)
base_model, checkpoint_payload = load_checkpoint(BASE_CKPT_DIR, device=DEVICE)

print(f"\nLoaded base model")
print(f"  Trained to epoch : {checkpoint_payload.get('epoch', 'unknown')}")
print(f"  Recorded metrics : {checkpoint_payload.get('metrics', {})}")
print(f"  Parameters       : {base_model.count_parameters():,}")
print(f"  Device           : {DEVICE}")

---

## Step 2 — Build the conditioned training format

Each question is rewritten with its control tokens in front. The model then
learns that the tokens *predict* the question that follows, which is what makes
them usable as controls at inference time.

In [ ]:
def load_split(name: str) -> list:
    path = SPLIT_DIR / f"{name}.jsonl"
    assert path.exists(), f"{path.name} missing — run Notebook 04."
    return [json.loads(line) for line in
            path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_records = load_split("train")
val_records = load_split("validation")

def format_conditioned(record: dict) -> str:
    """Control tokens first, then the question they should produce."""
    return (f"<DOMAIN: {record['domain']}> "
            f"<DIFFICULTY: {record['difficulty']}> "
            f"{record['question']}")

train_texts = [format_conditioned(r) for r in train_records]
val_texts = [format_conditioned(r) for r in val_records]

# Unconditioned text, for the like-for-like base comparison in Step 6.
train_plain = [r["question"] for r in train_records]
val_plain = [r["question"] for r in val_records]

print("CONDITIONED TRAINING FORMAT")
print("=" * 84)
for text in train_texts[:4]:
    print(f"  {text[:80]}")
print("=" * 84)

lengths = [len(tokenizer.encode(t)) for t in train_texts]
plain_lengths = [len(tokenizer.encode(t)) for t in train_plain]
MAX_SEQ_LEN = int(min(512, max(48, np.percentile(lengths, 99) + 4)))

print(f"\n  Mean tokens, plain question   : {np.mean(plain_lengths):.1f}")
print(f"  Mean tokens, with conditioning: {np.mean(lengths):.1f}")
print(f"  Control-token overhead        : "
      f"{np.mean(lengths) - np.mean(plain_lengths):.1f} tokens")
print(f"  Context window for this stage : {MAX_SEQ_LEN}")

In [ ]:
PAD_ID = 0

class ConditionedDataset(Dataset):
    """Causal-LM over conditioned text: control tokens then question."""

    def __init__(self, texts, tokenizer, max_len):
        self.samples = []
        self.truncated = 0
        for text in texts:
            ids = tokenizer.encode(text, add_special_tokens=True)
            if len(ids) > max_len:
                ids = ids[:max_len]
                self.truncated += 1
            if len(ids) >= 2:
                self.samples.append(ids)
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        ids = self.samples[index]
        padded = ids + [PAD_ID] * (self.max_len - len(ids))
        tensor = torch.tensor(padded, dtype=torch.long)
        targets = tensor.clone()
        targets[len(ids):] = -100
        return tensor[:-1], targets[1:]

BATCH_SIZE = 16 if DEVICE == "cuda" else 8

spec_train = ConditionedDataset(train_texts, tokenizer, MAX_SEQ_LEN)
spec_val = ConditionedDataset(val_texts, tokenizer, MAX_SEQ_LEN)
# Same data, unconditioned — the control group for Step 6.
plain_val = ConditionedDataset(val_plain, tokenizer, MAX_SEQ_LEN)

spec_train_loader = DataLoader(spec_train, batch_size=BATCH_SIZE, shuffle=True,
                               generator=torch.Generator().manual_seed(SEED))
spec_val_loader = DataLoader(spec_val, batch_size=BATCH_SIZE, shuffle=False)
plain_val_loader = DataLoader(plain_val, batch_size=BATCH_SIZE, shuffle=False)

print("SPECIALISATION DATA")
print("=" * 66)
print(f"  Training sequences        : {len(spec_train):,}")
print(f"  Validation (conditioned)  : {len(spec_val):,}")
print(f"  Validation (plain, control): {len(plain_val):,}")
print(f"  Batch size                : {BATCH_SIZE}")
print("=" * 66)

---

## Step 3 — Measure the base model *before* specialising

The baseline. Without it there is no way to tell whether specialisation helped,
and any later claim of improvement would be unfalsifiable.

The base model is evaluated on the conditioned validation text — the same data
the specialised model will be judged on — so the comparison is like for like.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

def evaluate(model, loader) -> float:
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            logits = model(inputs)
            if isinstance(logits, tuple):
                logits = logits[0]
            loss = criterion(logits.reshape(-1, logits.size(-1)),
                             targets.reshape(-1))
            n_tokens = int((targets != -100).sum())
            total_loss += float(loss) * n_tokens
            total_tokens += n_tokens
    return total_loss / max(total_tokens, 1)

# Keep an untouched copy of the base weights for the Step 6 comparison.
base_state = copy.deepcopy(base_model.state_dict())

base_conditioned_loss = evaluate(base_model, spec_val_loader)
base_plain_loss = evaluate(base_model, plain_val_loader)

print("BASELINE — the base model, before any specialisation")
print("=" * 72)
print(f"  On conditioned validation text : loss {base_conditioned_loss:.4f}  "
      f"(ppl {math.exp(min(base_conditioned_loss, 20)):9.2f})")
print(f"  On plain validation text       : loss {base_plain_loss:.4f}  "
      f"(ppl {math.exp(min(base_plain_loss, 20)):9.2f})")
print("=" * 72)
print("\nThe base model scores worse on the conditioned text because it has")
print("never seen the control tokens. Closing that gap is what Step 4 is for.")

---

## Step 4 — Continued training at a reduced learning rate

The learning rate is **⅕** of Stage 5's. Large updates to an already-trained
network overwrite what it learned — the failure mode known as catastrophic
forgetting — so specialisation proceeds in small steps and stops as soon as
validation loss stops improving.

In [ ]:
training_report = json.loads(
    (REPORTS_DIR / "candidate_training_report.json").read_text(encoding="utf-8"))
BASE_LR = training_report["hyperparameters"]["learning_rate"]

SPEC_LR = BASE_LR / 5.0
SPEC_EPOCHS = 6
SPEC_PATIENCE = 2
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

print(f"Stage 5 learning rate      : {BASE_LR:.2e}")
print(f"Specialisation rate (÷5)   : {SPEC_LR:.2e}")
print(f"Max epochs                 : {SPEC_EPOCHS}")
print(f"Early-stopping patience    : {SPEC_PATIENCE}")

model = base_model
optimizer = torch.optim.AdamW(model.parameters(), lr=SPEC_LR,
                              weight_decay=WEIGHT_DECAY)
total_steps = max(1, SPEC_EPOCHS * len(spec_train_loader))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                       T_max=total_steps)

SPEC_CKPT_DIR = CKPT_DIR / "specialized_model"
SPEC_CKPT_DIR.mkdir(parents=True, exist_ok=True)

history = {"epoch": [0], "train_loss": [np.nan],
           "val_loss": [base_conditioned_loss],
           "val_ppl": [float(math.exp(min(base_conditioned_loss, 20)))],
           "lr": [SPEC_LR], "seconds": [0.0]}

best_val = base_conditioned_loss     # must beat the base model to be saved
best_epoch = 0
epochs_without_gain = 0
run_start = time.perf_counter()

print(f"\n{'=' * 78}")
print("SPECIALISATION TRAINING")
print(f"  epoch 0 is the base model: val loss {base_conditioned_loss:.4f}")
print("=" * 78)

for epoch in range(1, SPEC_EPOCHS + 1):
    model.train()
    epoch_start = time.perf_counter()
    running_loss, running_tokens = 0.0, 0

    for inputs, targets in spec_train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        if isinstance(logits, tuple):
            logits = logits[0]
        loss = criterion(logits.reshape(-1, logits.size(-1)),
                         targets.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        n_tokens = int((targets != -100).sum())
        running_loss += float(loss) * n_tokens
        running_tokens += n_tokens

    train_loss = running_loss / max(running_tokens, 1)
    val_loss = evaluate(model, spec_val_loader)
    elapsed = time.perf_counter() - epoch_start

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_ppl"].append(float(math.exp(min(val_loss, 20))))
    history["lr"].append(optimizer.param_groups[0]["lr"])
    history["seconds"].append(elapsed)

    marker = ""
    if val_loss < best_val - 1e-4:
        best_val, best_epoch = val_loss, epoch
        epochs_without_gain = 0
        save_checkpoint(SPEC_CKPT_DIR, model, optimizer, scheduler,
                        epoch=epoch, step=epoch * len(spec_train_loader),
                        metrics={"val_loss": val_loss,
                                 "train_loss": train_loss,
                                 "base_val_loss": base_conditioned_loss,
                                 "improvement": base_conditioned_loss - val_loss})
        marker = "  <- best, checkpoint saved"
    else:
        epochs_without_gain += 1

    print(f"  epoch {epoch}/{SPEC_EPOCHS}  train {train_loss:.4f}  "
          f"val {val_loss:.4f} (ppl {history['val_ppl'][-1]:9.2f})  "
          f"{elapsed:5.1f}s{marker}")

    if epochs_without_gain >= SPEC_PATIENCE:
        print(f"  Early stopping: no improvement for {SPEC_PATIENCE} epochs.")
        break

spec_seconds = time.perf_counter() - run_start
SPECIALISATION_HELPED = best_epoch > 0

print("=" * 78)
if SPECIALISATION_HELPED:
    print(f"Specialisation improved validation loss: "
          f"{base_conditioned_loss:.4f} -> {best_val:.4f} "
          f"({base_conditioned_loss - best_val:+.4f})")
else:
    print("Specialisation did NOT beat the base model on validation loss.")
    print("No checkpoint was saved. Stage 8 will evaluate the BASE model.")
print("=" * 78)

---

## Step 5 — Specialisation learning curve

Epoch 0 is the base model, so the first segment of the curve shows what
specialisation bought immediately. A curve that rises after an early dip is
catastrophic forgetting in progress — the marker shows where early stopping cut
it off.

In [ ]:
# ── Figure 7.1 — specialisation curve ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

epochs = history["epoch"]
axes[0].plot(epochs, history["val_loss"], marker="s", markersize=6,
             linewidth=2.3, color=PALETTE[3], label="validation loss")
train_epochs = [e for e, v in zip(epochs, history["train_loss"])
                if not np.isnan(v)]
train_values = [v for v in history["train_loss"] if not np.isnan(v)]
axes[0].plot(train_epochs, train_values, marker="o", markersize=5,
             linewidth=2.1, color=PALETTE[0], label="training loss")
axes[0].axhline(base_conditioned_loss, color=PALETTE[4], linestyle="--",
                linewidth=1.8,
                label=f"base model = {base_conditioned_loss:.4f}")
if SPECIALISATION_HELPED:
    axes[0].axvline(best_epoch, color=PALETTE[2], linestyle=":", linewidth=2,
                    label=f"best epoch {best_epoch}")
axes[0].set_title("Specialisation loss (epoch 0 = base model)")
axes[0].set_xlabel("Specialisation epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend(fontsize=8)

axes[1].plot(epochs, history["val_ppl"], marker="D", markersize=5.5,
             linewidth=2.2, color=PALETTE[1])
axes[1].axhline(math.exp(min(base_conditioned_loss, 20)), color=PALETTE[4],
                linestyle="--", linewidth=1.8, label="base perplexity")
axes[1].set_yscale("log")
axes[1].set_title("Validation perplexity (log scale)")
axes[1].set_xlabel("Specialisation epoch")
axes[1].set_ylabel("Perplexity")
axes[1].legend(fontsize=8)

# Improvement relative to the base model, per epoch.
improvement = [base_conditioned_loss - v for v in history["val_loss"]]
colours = [PALETTE[2] if i > 0 else PALETTE[3] for i in improvement]
bars = axes[2].bar(epochs, improvement, color=colours, width=0.62)
axes[2].axhline(0, color="black", linewidth=1.2)
axes[2].set_title("Improvement over the base model")
axes[2].set_xlabel("Specialisation epoch")
axes[2].set_ylabel("Loss reduction (positive = better)")
axes[2].bar_label(bars, fmt="%+.3f", padding=2, fontsize=7.5)
axes[2].margins(y=0.2)

fig.suptitle(f"Specialising {selected['label']} — continued training at "
             f"LR/5", y=1.03, fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "specialisation_curve",
            "Green bars in the right panel are epochs that beat the base "
            "model; red bars are epochs that made it worse.")
plt.show()

---

## Step 6 — Did specialisation actually help? Base vs specialised

The verdict. Both models are evaluated on **both** validation formats:

- **Conditioned text** — the task the runtime needs.
- **Plain text** — the base model's original task, which reveals whether
  general language ability was sacrificed to gain task performance.

The second column is what makes this a real test of catastrophic forgetting
rather than a one-sided comparison.

In [ ]:
# Restore the base weights into a clean model for a fair comparison.
comparison_base = build_candidate_model(
    CANDIDATE_ID, vocab_size=tokenizer.vocab_size
    if hasattr(tokenizer, "vocab_size") else 4096).to(DEVICE)
try:
    comparison_base.load_state_dict(base_state)
    base_restored = True
except Exception as exc:
    print(f"Note: could not rebuild the base model for comparison ({exc}); "
          f"using the recorded baseline losses instead.")
    base_restored = False

if SPECIALISATION_HELPED:
    specialised_model, _ = load_checkpoint(SPEC_CKPT_DIR, device=DEVICE)
else:
    specialised_model = model

results = {
    "base": {
        "conditioned": (evaluate(comparison_base, spec_val_loader)
                        if base_restored else base_conditioned_loss),
        "plain": (evaluate(comparison_base, plain_val_loader)
                  if base_restored else base_plain_loss),
    },
    "specialised": {
        "conditioned": evaluate(specialised_model, spec_val_loader),
        "plain": evaluate(specialised_model, plain_val_loader),
    },
}

comparison = pd.DataFrame([
    {
        "model": name.title(),
        "conditioned_loss": round(values["conditioned"], 4),
        "conditioned_ppl": round(math.exp(min(values["conditioned"], 20)), 2),
        "plain_loss": round(values["plain"], 4),
        "plain_ppl": round(math.exp(min(values["plain"], 20)), 2),
    }
    for name, values in results.items()
])

print("BASE vs SPECIALISED — validation split")
print("=" * 88)
print(comparison.to_string(index=False))
print("=" * 88)

task_gain = results["base"]["conditioned"] - results["specialised"]["conditioned"]
general_change = results["base"]["plain"] - results["specialised"]["plain"]

print(f"\n  Task performance (conditioned text) : {task_gain:+.4f} "
      f"({'improved' if task_gain > 0 else 'degraded'})")
print(f"  General ability  (plain text)       : {general_change:+.4f} "
      f"({'retained/improved' if general_change > -0.05 else 'DEGRADED'})")

print("\nVERDICT")
print("-" * 72)
if task_gain > 0.01 and general_change > -0.10:
    PROMOTE_SPECIALISED = True
    print("  Specialisation SUCCEEDED. Task performance improved without")
    print("  meaningfully sacrificing general language ability.")
    print("  -> Stage 8 evaluates the SPECIALISED model.")
elif task_gain > 0.01:
    PROMOTE_SPECIALISED = True
    print("  Specialisation improved the task but degraded general ability")
    print(f"  by {abs(general_change):.4f}. This is catastrophic forgetting.")
    print("  -> Stage 8 evaluates the SPECIALISED model, and this trade-off")
    print("     is recorded as a limitation in the specialisation report.")
else:
    PROMOTE_SPECIALISED = False
    print("  Specialisation did NOT improve task performance.")
    print("  -> Stage 8 evaluates the BASE model. Reporting the specialised")
    print("     model as an improvement would be false.")
print("-" * 72)

In [ ]:
# ── Figure 7.2 — base vs specialised ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

x = np.arange(2)
width = 0.36
base_values = [results["base"]["conditioned"], results["base"]["plain"]]
spec_values = [results["specialised"]["conditioned"],
               results["specialised"]["plain"]]

b1 = axes[0].bar(x - width / 2, base_values, width, label="base",
                 color=PALETTE[4])
b2 = axes[0].bar(x + width / 2, spec_values, width, label="specialised",
                 color=PALETTE[2])
axes[0].set_xticks(x)
axes[0].set_xticklabels(["Conditioned\n(the runtime task)",
                         "Plain\n(general ability)"])
axes[0].set_ylabel("Validation cross-entropy loss")
axes[0].set_title("Base vs specialised — lower is better")
axes[0].legend(fontsize=9)
axes[0].bar_label(b1, fmt="%.3f", padding=2, fontsize=8)
axes[0].bar_label(b2, fmt="%.3f", padding=2, fontsize=8)
axes[0].margins(y=0.16)

changes = [task_gain, general_change]
labels = ["Task\n(conditioned)", "General\n(plain)"]
colours = [PALETTE[2] if c > 0 else PALETTE[3] for c in changes]
bars = axes[1].bar(labels, changes, color=colours, width=0.5)
axes[1].axhline(0, color="black", linewidth=1.2)
axes[1].set_title("Change from specialisation\n(positive = improvement)")
axes[1].set_ylabel("Loss reduction")
axes[1].bar_label(bars, fmt="%+.4f", padding=3, fontsize=9)
axes[1].margins(y=0.28)
axes[1].annotate(
    "A negative bar here would be\ncatastrophic forgetting",
    xy=(0.97, 0.06), xycoords="axes fraction", ha="right", fontsize=8,
    style="italic",
    bbox=dict(boxstyle="round,pad=0.35", fc="#FFF4E5", ec=PALETTE[1]))

fig.suptitle("Did specialisation help, and at what cost?", y=1.03,
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "base_vs_specialised",
            "The plain-text bar is the forgetting check: task gains that come "
            "at the cost of general ability are visible here.")
plt.show()

---

## Step 7 — Qualitative check: what does it actually generate?

Loss is a proxy. This step generates from the model with real control tokens and
prints the output, because a model can improve its loss while still producing
unusable text.

**On a corpus this small, expect the output to be weak.** Reporting that plainly
is more useful than a metric that hides it.

In [ ]:
def generate_question(model, domain: str, difficulty: str,
                      max_new_tokens: int = 28,
                      temperature: float = 0.8) -> str:
    """Generate with the control tokens as the prompt."""
    prompt = f"<DOMAIN: {domain}> <DIFFICULTY: {difficulty}>"
    ids = tokenizer.encode(prompt, add_special_tokens=True)
    tensor = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        output = model.generate(tensor, max_new_tokens=max_new_tokens,
                                temperature=temperature, top_k=40)
    text = tokenizer.decode(output[0].tolist(), skip_special_tokens=True)
    for token in (f"<DOMAIN: {domain}>", f"<DIFFICULTY: {difficulty}>",
                  "DOMAIN", "DIFFICULTY", domain, difficulty):
        text = text.replace(token, " ")
    return " ".join(text.split()).strip()

PROMPTS = [
    ("SQL", "Beginner"),
    ("OOP", "Intermediate"),
    ("System Design", "Advanced"),
    ("Frontend Development", "Intermediate"),
]

print("QUALITATIVE GENERATION SAMPLES")
print("=" * 84)
samples = []
for domain, difficulty in PROMPTS:
    text = generate_question(specialised_model, domain, difficulty)
    samples.append({"domain": domain, "difficulty": difficulty,
                    "generated": text})
    print(f"\n  [{domain} / {difficulty}]")
    print(f"    {text[:150] if text else '(empty output)'}")
print("\n" + "=" * 84)

# An honest, measurable readout of output quality.
from collections import Counter
quality = []
for sample in samples:
    text = sample["generated"]
    words = text.split()
    quality.append({
        "domain": sample["domain"],
        "words": len(words),
        "distinct_word_ratio": round(len(set(w.lower() for w in words))
                                     / max(len(words), 1), 3),
        "is_question_shaped": bool(text.endswith("?") or
                                   text.lower().startswith(
                                       ("what", "why", "how", "when",
                                        "explain", "describe", "which"))),
    })

quality_df = pd.DataFrame(quality)
print("\nOUTPUT QUALITY READOUT")
print(quality_df.to_string(index=False))

usable = int(quality_df["is_question_shaped"].sum())
print(f"\n  Question-shaped outputs: {usable} / {len(quality_df)}")
print(f"  Mean distinct-word ratio: "
      f"{quality_df['distinct_word_ratio'].mean():.3f} "
      f"(low values indicate repetition)")
print("\nHONEST ASSESSMENT")
print("-" * 76)
if usable < len(quality_df) / 2 or quality_df["distinct_word_ratio"].mean() < 0.5:
    print("  Generation quality is POOR. A Transformer trained from scratch on")
    print("  a corpus of this size cannot produce fluent questions — there is")
    print("  simply not enough text to learn the grammar of the domain.")
    print()
    print("  This is why the RUNTIME uses this model as one source among two:")
    print("  the ML service tries the model first and falls back to retrieval")
    print("  over the labelled question dataset when the generated question")
    print("  fails a quality gate (see `interview_engine.QuestionPool`).")
    print("  The interview a user experiences is therefore never degraded by")
    print("  this limitation, and the limitation itself is not hidden.")
else:
    print("  Generation is producing question-shaped output. Even so, the")
    print("  runtime keeps retrieval as a fallback, because a single weak")
    print("  generation should never reach a live candidate.")
print("-" * 76)

---

## Step 8 — Specialisation report

In [ ]:
specialisation_report = {
    "stage": "07_model_specialisation",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "policy": "Continued training of this project's own Stage 5 weights. "
              "No LoRA, adapters, or external fine-tuning frameworks.",
    "base_model": {
        "candidate_id": CANDIDATE_ID,
        "label": selected["label"],
        "checkpoint": selected["checkpoint"],
        "parameters": int(selected["parameters"]),
    },
    "objective": {
        "format": "<DOMAIN: {domain}> <DIFFICULTY: {difficulty}> {question}",
        "purpose": "condition generation on the requested domain and difficulty",
        "control_token_overhead_tokens": round(
            float(np.mean(lengths) - np.mean(plain_lengths)), 2),
    },
    "hyperparameters": {
        "base_learning_rate": BASE_LR,
        "specialisation_learning_rate": SPEC_LR,
        "reduction_factor": 5,
        "max_epochs": SPEC_EPOCHS,
        "patience": SPEC_PATIENCE,
        "weight_decay": WEIGHT_DECAY,
        "grad_clip": GRAD_CLIP,
        "rationale": "LR reduced to guard against catastrophic forgetting",
    },
    "history": history,
    "epochs_run": len(history["epoch"]) - 1,
    "best_epoch": best_epoch,
    "comparison": {
        "base_conditioned_loss": round(results["base"]["conditioned"], 5),
        "specialised_conditioned_loss": round(
            results["specialised"]["conditioned"], 5),
        "base_plain_loss": round(results["base"]["plain"], 5),
        "specialised_plain_loss": round(results["specialised"]["plain"], 5),
        "task_improvement": round(task_gain, 5),
        "general_ability_change": round(general_change, 5),
        "catastrophic_forgetting_detected": bool(general_change < -0.10),
    },
    "verdict": {
        "specialisation_helped": bool(SPECIALISATION_HELPED),
        "promote_specialised": bool(PROMOTE_SPECIALISED),
        "model_for_stage_8": ("specialised" if PROMOTE_SPECIALISED else "base"),
        "checkpoint_for_stage_8": str(
            (SPEC_CKPT_DIR if PROMOTE_SPECIALISED else BASE_CKPT_DIR)
            .relative_to(WORKSPACE_DIR)),
    },
    "generation_samples": samples,
    "generation_quality": quality_df.to_dict(orient="records"),
    "known_limitation": (
        "The training corpus is small for from-scratch language modelling, so "
        "generation fluency is limited. The runtime mitigates this by gating "
        "generated questions on a quality check and falling back to retrieval "
        "over the labelled dataset."
    ),
    "train_seconds": round(spec_seconds, 1),
    "device": DEVICE,
}

report_path = REPORTS_DIR / "specialization_report.json"
report_path.write_text(json.dumps(specialisation_report, indent=2, default=str),
                       encoding="utf-8")

print(f"Specialisation report : {report_path.relative_to(WORKSPACE_DIR)}")
print(f"Model for Stage 8     : "
      f"{specialisation_report['verdict']['model_for_stage_8'].upper()}")
print(f"Checkpoint            : "
      f"{specialisation_report['verdict']['checkpoint_for_stage_8']}")

---

## Stage 7 summary

| Aspect | Approach |
|---|---|
| Weight provenance | asserted to be this project's Stage 5 checkpoint |
| Method | direct continued training — no LoRA, no adapters |
| Learning rate | reduced to ⅕ of Stage 5's |
| Forgetting control | early stopping + plain-text control evaluation |
| Success criterion | must beat the base model on validation, or it is not promoted |
| Generation quality | measured and reported, including when poor |

### The two results that matter

1. **Task improvement** — did conditioned validation loss fall? Step 6 answers
   this and refuses to promote the specialised model if it did not.
2. **Forgetting check** — did plain-text loss rise? Figure 7.2's second bar
   makes any trade-off visible instead of leaving it unmeasured.

### The limitation, stated plainly

A Transformer trained from scratch on a corpus of this size cannot generate
fluent interview questions. Step 7 measures that rather than asserting success.
The runtime handles it by gating generated questions on a quality check and
falling back to retrieval over the labelled dataset — so the limitation is
mitigated in the product and disclosed in the research.

### Next

**Notebook 08 — Held-Out Test Evaluation**, the single authorised opening of the
sealed test split.